In [1]:
!pip install faiss-cpu openai


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\r2com\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import json
import numpy as np
import faiss
import os
from dotenv import load_dotenv
from openai import OpenAI

# 모든 JSON 데이터 로드
json_files = [f for f in os.listdir("C:/Users/r2com/Documents/GitHub/BeBaek/ChefBot/data/recipes") if f.endswith(".json")]

all_recipes = []
for file in json_files:
    with open(f"C:/Users/r2com/Documents/GitHub/BeBaek/ChefBot/data/recipes/{file}", "r", encoding="utf-8") as f:
        data = json.load(f)
        all_recipes.extend(data)

print(f"총 {len(all_recipes)} 개의 레시피 데이터 로드 완료")

# OpenAI API 키 설정
API_KEY = os.getenv("OPENAI_API_KEY") 
client = OpenAI(api_key=API_KEY)

# 벡터화할 텍스트 데이터 준비
texts = [f"{recipe['name']} {recipe['ingredients']} {' '.join(recipe['recipe'])}" for recipe in all_recipes]
# texts = [f"{recipe['name']} {recipe['ingredients']}" for recipe in all_recipes]

# OpenAI Embedding 생성 (Batch 처리)
def get_embeddings(texts, batch_size=100):
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        try:
            response = client.embeddings.create(
                input=batch, 
                model="text-embedding-ada-002"
            )
            batch_embeddings = [item.embedding for item in response.data] 
            embeddings.extend(batch_embeddings)
        except Exception as e:
            print(f"OpenAI API 호출 오류 발생: {e}")

    return np.array(embeddings)

# 모든 레시피 데이터 임베딩 생성
embeddings = get_embeddings(texts)

# FAISS 벡터 DB 구축
d = embeddings.shape[1]  # 벡터 차원 수
index = faiss.IndexFlatL2(d)  # L2 거리 기반 벡터 검색 인덱스
index.add(embeddings)  # 벡터 추가

# FAISS 인덱스 저장
faiss.write_index(index, "./faiss_index/recipes_faiss.index")

print("벡터화 완료 & FAISS 저장 완료")

총 3900 개의 레시피 데이터 로드 완료
벡터화 완료 & FAISS 저장 완료


In [20]:
import faiss

# 저장된 FAISS 인덱스 로드
index_path = "./faiss_index/recipes_faiss.index"
index = faiss.read_index(index_path)

# 벡터 개수 확인
print(f"FAISS 인덱스 로드 완료, 저장된 벡터 개수: {index.ntotal}")

FAISS 인덱스 로드 완료, 저장된 벡터 개수: 3900


In [21]:
import numpy as np
import faiss

# 저장된 FAISS 인덱스 로드
index = faiss.read_index(index_path)

# 테스트용 랜덤 벡터 생성 (벡터 차원 수 맞추기)
d = index.d  # 벡터 차원 수
test_vector = np.random.random((1, d)).astype('float32')

# 가장 가까운 3개 벡터 찾기
D, I = index.search(test_vector, 3)

print(f"검색된 벡터의 인덱스: {I[0]}")
print(f"검색된 벡터의 거리: {D[0]}")


검색된 벡터의 인덱스: [1640 1966  756]
검색된 벡터의 거리: [512.32825 512.4159  512.41864]


In [22]:
# 검색된 레시피 확인
search_results = [all_recipes[i] for i in I[0]]

print("검색된 레시피:")
for i, recipe in enumerate(search_results):
    print(f"{i+1}. {recipe['name']} - 재료: {recipe['ingredients']} - 레시피: {recipe['recipe']}")

검색된 레시피:
1. 보리차 - 재료: 겉보리, 물, PN멀티팟 - 레시피: ['1. 보리를 깨끗이 씻어서 물기를 제거해주세요.', '2. 물기를 잘 말린 보리를 알곡 색이 갈색으로 변할 때까지 약불에서 은근히 볶아주세요.', '3. 볶기 전 보리(왼쪽)와 약불에서 살살 익힌 보리(오른쪽)!', '4. 볶아낸 보리를 채망에 담아 물을 붓고 끓여주세요.', '5. 다 끓인 보리차는 따뜻하게 드셔도 좋고, 병에 옮겨 담아 냉장보관하셔서 시원하게 드셔도 좋답니다.']
2. 보석 같은 석류 알 쉽게 빼내는 비법 - 재료: 석류 - 레시피: ['1. 석류는 1/2로 잘라준다.', '2. 옆으로 비틀어 준 후 숟가락으로 두드려주고 남은 건 긁어낸다.', '3. 알과 함께 떨어진 심지는 가려서 빼준다.']
3. 대추차 - 재료: 대추 10줌, 물 - 레시피: ['1. 대추 10줌에 물은 자작하게 넣고 끓여주세요.', '2. 끓으면서 대추가 빵빵하게 부풀어 올라 냄비 밖으로 튀어나오려고 하니 숨을 가라앉히기 위해 몇번씩 휘저으면서 터트려주세요.', '3. 국물은 냄비에 그대로 걸러두고, 대추를 채에 걸러서 누르고 비벼가며 내려줍니다.', '4. 암튼 처음 대추 끓였던 물에 채에 내린 대추를 합쳐서 불에 올려 팔팔 30분정도 끓여줍니다. 이때 설탕을 넣는 경우도 있는데 저는 대추만으로 충분히 달아 아무것도 안넣었어요. 더 단맛이 좋은분들은 나중에 차로 마실때 꿀을 넣어 마시는게 좋을 것 같네요.', '5. 유리병은 끓는물에 소독해서 끓인 대추차를 넣어 마개를 꼭 닫아 뒤집어서 충분히 식힌후에 냉장보관하면 끝!']


In [23]:
print(f"총 저장된 FAISS 벡터 개수: {index.ntotal}")

# 첫 번째 벡터 출력
first_vector = index.reconstruct(0)
print("첫 번째 벡터:", first_vector[:50])  # 10개만 출력

총 저장된 FAISS 벡터 개수: 3900
첫 번째 벡터: [ 0.00425692 -0.01013179  0.00057941 -0.03736952 -0.01044494 -0.01032751
 -0.03611691 -0.01677975 -0.046738   -0.01911535 -0.02111169  0.02051148
 -0.04269312 -0.00195231 -0.01377871  0.03504698  0.01337422 -0.00078329
  0.00671321 -0.00562696 -0.00502675 -0.00868346 -0.01447025  0.00023446
  0.01539666  0.00212846  0.02411274 -0.00584225 -0.00765266  0.01928497
  0.02794885 -0.01467902 -0.01068633  0.0059238  -0.00085954  0.01543581
 -0.00170603 -0.01267615  0.01937631 -0.00669363 -0.00882698 -0.0017729
  0.01851514  0.01032098  0.02266441  0.01267615  0.01957203 -0.02022443
 -0.01026879  0.01958507]
